# Exploration du graphe des aéroports
Avant de construire les modèles, on cherche à comprendre de quoi est fait le graphe :
ce que contiennent les nœuds et les arêtes, comment le réseau est structuré,
et quelles informations pourront servir à prédire des routes.

In [ ]:
import networkx as nx  # biblio qui permet de lire un fichier .graphml et calculer des choses sur un graphe
import numpy as np

# Chargement du graphml et construction d'un objet graphe avec id int
G = nx.read_graphml("data/raw/airports.graphml.xml", node_type=int)
print(G)

In [ ]:
# on veut essayer de comprendre le graphe qu'on a recupéré

# Les attributs de quelques villes (nœuds)
for node in list(G.nodes(data=True))[:3]:
    print(node)

# Les attributs de quelques routes (arêtes)
for edge in list(G.edges(data=True))[:3]:
    print(edge)

In [ ]:
# Statistiques de base
print("Orienté :", G.is_directed())
print("Composantes connexes :", nx.number_connected_components(G))
print("Clustering moyen :", nx.average_clustering(G))

# on regarde les noeuds avec le plus de voisins (degré)
degres = dict(G.degree())
noms = nx.get_node_attributes(G, "city_name")
top10 = sorted(degres.items(), key=lambda t: -t[1])[:10]
print("Top 10 hubs :", [(noms[n], d) for n, d in top10])

# on compare avec un aéroport "typique"
print("Degré médian :", np.median(list(degres.values())))
print("Villes avec une seule route :", sum(d == 1 for d in degres.values()))

In [ ]:
# distribution des composantes connexes : le graphe est-il "connecté" ?
composantes = sorted(nx.connected_components(G), key=len, reverse=True)
print("Tailles :", [len(c) for c in composantes])

# qui sont les villes hors de la grande composante ?
for c in composantes[1:]:
    print([noms[n] for n in c])

In [ ]:
# on etudie la distribution de la population
pops = np.array([d["population"] for _, d in G.nodes(data=True)])

print("Population min     :", pops.min())
print("Population médiane :", np.median(pops))
print("Population moyenne :", round(pops.mean()))
print("Population max     :", pops.max())
print("Villes à exactement 10 000 habitants :", (pops == 10000).sum(), "/", len(pops))

In [ ]:
# structure vs géographie : les routes existantes sont-elles différentes de paires au hasard ?
import random
random.seed(0)

def distance_km(u, v):
    a, b = G.nodes[u], G.nodes[v]
    la1, lo1, la2, lo2 = map(np.radians, [a["lat"], a["lon"], b["lat"], b["lon"]])
    h = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(h))

routes = random.sample(list(G.edges()), 3000)
noeuds = list(G.nodes())
hasard = []
while len(hasard) < 3000:
    u, v = random.sample(noeuds, 2)
    if not G.has_edge(u, v):
        hasard.append((u, v))

cn = lambda paires: np.array([len(list(nx.common_neighbors(G, u, v))) for u, v in paires])
dist = lambda paires: np.array([distance_km(u, v) for u, v in paires])

print("Voisins communs (médiane)  routes :", np.median(cn(routes)), "| hasard :", np.median(cn(hasard)))
print("Distance en km (médiane)   routes :", round(np.median(dist(routes))), "| hasard :", round(np.median(dist(hasard))))

In [ ]:
# à distance égale, deux villes du même pays sont-elles plus souvent reliées ?
lat = np.radians([G.nodes[n]["lat"] for n in noeuds])
lon = np.radians([G.nodes[n]["lon"] for n in noeuds])
pays = np.array([G.nodes[n]["country"] for n in noeuds])
A = nx.to_numpy_array(G, nodelist=noeuds)
h = np.sin((lat[:, None] - lat) / 2) ** 2 + np.cos(lat)[:, None] * np.cos(lat) * np.sin((lon[:, None] - lon) / 2) ** 2
D = 6371 * 2 * np.arcsin(np.sqrt(np.clip(h, 0, 1)))
iu = np.triu_indices(len(noeuds), 1)
D, S, A = D[iu], (pays[:, None] == pays)[iu], A[iu]
for lo, hi in [(0, 500), (500, 1000), (1000, 2000)]:
    m = (D >= lo) & (D < hi)
    print(f"{lo}-{hi} km : même pays {A[m & S].mean():.1%} | pays différents {A[m & ~S].mean():.1%}")

D'après l'exploration du graphe :
- Le graphe n'est pas orienté : chaque route relie deux villes sans sens de circulation. Lors du découpage train/test, une route cachée doit l'être dans les deux sens.
- Chaque nœud correspond à une ville et non à un aéroport : les aéroports d'une même métropole sont regroupés (par exemple Paris), ce qui renforce le degré des grandes villes.
- La grande majorité des villes (3 351 sur 3 363) forment une seule composante connexe. Les autres forment de petits groupes reliés uniquement entre eux. Pour ces villes, la structure du réseau ne peut pas aider à prédire une route vers le reste du monde. Comme elles sont très peu nombreuses, on les garde.
- Le clustering moyen est élevé (0,49) : quand une ville est reliée à deux autres, ces deux-là sont souvent reliées entre elles. Les méthodes basées sur les voisins communs devraient donc déjà bien fonctionner.
- Le réseau est organisé autour de hubs : la médiane est de 3 routes par ville, mais les plus grandes en ont plus de 200. Le degré est donc une information très prédictive, mais c'est aussi un risque : un modèle pourrait obtenir un bon score simplement en reliant les villes aux hubs. On s'attend à ce que les routes entre petites villes soient les plus difficiles à retrouver, en particulier pour celles qui n'ont qu'une seule route : si cette route est cachée dans le test, la ville n'a plus aucun voisin à l'entraînement, et seules ses features peuvent aider.
- La distribution de la population est très déséquilibrée : la moyenne (338 078 habitants) est plus de 20 fois supérieure à la médiane (15 061). Il faudra une transformation (logarithme) pour que les grandes villes n'écrasent pas les autres. De plus, la moitié des villes ont exactement 10 000 habitants, probablement une valeur par défaut utilisée quand l'information manquait : cette feature est bruitée.

In [ ]:
# bibliothèque pour tracer des figures
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# Position de chaque ville : (longitude, latitude) → la carte du monde apparaît
pos = {n: (d["lon"], d["lat"]) for n, d in G.nodes(data=True)}

# On trie les villes du plus petit au plus grand degré :
# les hubs sont dessinés en dernier, donc au-dessus des autres
noeuds_tries = sorted(G.nodes(), key=lambda n: degres[n])
x = [pos[n][0] for n in noeuds_tries]
y = [pos[n][1] for n in noeuds_tries]
deg = [degres[n] for n in noeuds_tries]

fig, ax = plt.subplots(figsize=(14, 7))

# 1) Les routes, en gris très clair, en arrière-plan
nx.draw_networkx_edges(G, pos=pos, ax=ax, width=0.05, edge_color="lightgray")

# 2) Les villes : couleur ET taille selon le nombre de routes
sc = ax.scatter(x, y, c=deg, s=[2 + d / 3 for d in deg],
                cmap="plasma", norm=LogNorm(), zorder=2)
plt.colorbar(sc, ax=ax, label="Nombre de routes (échelle log)")

# 3) Le top 10 des hubs dans un encadré, sur deux colonnes de 5
top10 = sorted(degres, key=degres.get, reverse=True)[:10]
etiquettes = [f"{noms[n]} : {degres[n]}" for n in top10]
lignes = [f"{etiquettes[i]:<24}{etiquettes[i + 5]}" for i in range(5)]
texte = "Top 10 hubs\n" + "\n".join(lignes)
ax.text(0.45, 0.02, texte, transform=ax.transAxes, fontsize=8, va="bottom",
        family="monospace",
        bbox=dict(facecolor="white", edgecolor="gray", alpha=0.9))

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Réseau aérien mondial : couleur et taille selon le nombre de routes")
plt.show()

La carte montre que les hubs (en jaune et orange) sont concentrés en Europe de l'Ouest et aux États-Unis. Le reste du monde est surtout composé de petites villes (en violet), reliées aux grands réseaux par quelques hubs régionaux.

### Hypothèses issues de l'exploration

Pour prédire si une route existe entre deux villes, le modèle dispose de deux sources d'information :
- **les features** : ce que l'on sait de chaque ville prise isolément (position, population, pays) ;
- **la structure** : la façon dont les villes sont déjà reliées entre elles (voisins communs, hubs).

L'exploration nous amène à formuler les hypothèses suivantes :

- **H1 – Les heuristiques seront difficiles à battre.** Le réseau forme beaucoup de triangles (clustering moyen de 0,49), et les paires de villes reliées ont en médiane 6 voisins communs, contre 0 pour des paires tirées au hasard.
- **H2 – Les features géographiques sont très prédictives.** Les routes existantes relient des villes distantes en médiane d'environ 800 km, contre environ 9 000 km pour des paires au hasard. Un modèle qui n'utilise que les features devrait donc lui aussi bien fonctionner, en particulier face à des négatifs aléatoires, souvent très éloignés.
- **H3 – Le pays apporte une information que la distance ne donne pas.** À distance égale, deux villes du même pays ont environ deux fois plus de chances d'être reliées.
- **H4 – Combiner structure et features devrait faire mieux que chacune seule.** C'est ce que permet le message passing d'un GNN.

### Choix des features

Nous utilisons tous les attributs informatifs, puis nous mesurons l'importance de chacun par une étude d'ablation :

- **Identifiant du nœud** : sert uniquement à identifier chaque ville, ce n'est pas une feature.
- **Latitude, longitude** : converties en coordonnées 3D sur la sphère (x, y, z). Cela évite qu'une longitude de 179° et une de -179°, pourtant voisines, apparaissent très éloignées.
- **Population** : très déséquilibrée (de 637 à 22 millions d'habitants) ; nous utilisons son logarithme, normalisé. Elle est bruitée (valeur par défaut de 10 000 pour la moitié des villes).
- **Pays** : à distance égale, deux villes du même pays sont environ deux fois plus souvent reliées. Il est encodé en one-hot (212 colonnes), et non avec un simple numéro, qui inventerait un ordre entre les pays.
- **Nom de ville** : simple libellé, non unique (plusieurs villes portent le même nom). Il n'est pas utilisé comme feature, mais sert à afficher les résultats.

Chaque ville est finalement représentée par **216 features** : 3 coordonnées, 1 pour la population et 212 pour le pays.